**Analista**: Emilio Caldas


# 📊 Proyecto Final: Desafío de Negocio Comercial Andina

**Análisis Predictivo de Ventas Semanales para una Empresa de Retail**

Este notebook desarrolla un análisis predictivo completo para Comercial Andina, una empresa de retail que busca anticipar sus ventas semanales. El proyecto incluye:

1. **Preparación de datos**: Limpieza y transformación de datos históricos de ventas (fecha, producto, sucursal, cantidad, precio).
2. **Modelo predictivo**: Entrenamiento de un modelo de regresión lineal para predecir las ventas.
3. **Prueba de hipótesis**: Verificación estadística de si las promociones influyen significativamente en las ventas.
4. **Conclusiones y recomendaciones**: Visualizaciones, hallazgos y recomendaciones para optimizar decisiones comerciales.

**Dataset**: `workspace.comercial_andina.comercial_andina_2_m` (2,000,000 transacciones, 6 sucursales, 5 categorías, año 2025)

In [0]:
#%pip install plotly scikit-learn

In [0]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
pio.templates.default = "plotly_white"

In [0]:
# Cargar datos desde la tabla de Spark
df_spark = spark.table("workspace.comercial_andina.comercial_andina_2_m")
df = df_spark.toPandas()
print(f"Total de registros cargados: {len(df):,}")
print(f"Columnas: {list(df.columns)}")
display(df.head(10))

Total de registros cargados: 2,000,000
Columnas: ['id_transaccion', 'fecha', 'sucursal', 'categoria', 'ventas', 'precio_unitario', 'promo', 'descuento_pct']


id_transaccion,fecha,sucursal,categoria,ventas,precio_unitario,promo,descuento_pct
1,2025-09-15,Quito-Norte,Hogar,61.93,141.22,0,0.0
2,2025-10-10,Guayaquil-Centro,Cuidado Personal,122.08,31.42,1,39.6
3,2025-03-19,Guayaquil-Centro,Limpieza,143.58,25.14,1,28.4
4,2025-11-30,Ambato-Centro,Alimentos,58.55,96.7,0,0.0
5,2025-01-30,Guayaquil-Centro,Hogar,138.54,86.27,1,16.9
6,2025-04-27,Ambato-Centro,Hogar,84.58,288.7,1,10.4
7,2025-01-29,Cuenca-Este,Alimentos,139.81,264.09,1,11.6
8,2025-05-11,Ambato-Centro,Hogar,47.07,155.38,0,0.0
9,2025-07-25,Guayaquil-Centro,Alimentos,141.94,176.13,0,null
10,2025-01-28,Ambato-Centro,Alimentos,14.63,238.08,1,11.5


In [0]:
print("=" * 60)
print("📊 INFORMACIÓN GENERAL DEL DATASET")
print("=" * 60)
print(f"\nTotal de registros: {len(df):,}")
print(f"Total de columnas: {df.shape[1]}")
print(f"\nRango de fechas: {df['fecha'].min()} → {df['fecha'].max()}")
print(f"\nSucursales ({df['sucursal'].nunique()}): {sorted(df['sucursal'].unique())}")
print(f"Categorías ({df['categoria'].nunique()}): {sorted(df['categoria'].unique())}")
print(f"\nTransacciones con promoción (promo=1): {(df['promo']==1).sum():,}")
print(f"Transacciones sin promoción (promo=0): {(df['promo']==0).sum():,}")
print(f"\n{'─'*60}")
print("Estadísticas descriptivas:")
display(df.describe())

📊 INFORMACIÓN GENERAL DEL DATASET

Total de registros: 2,000,000
Total de columnas: 8

Rango de fechas: 2025-01-01 → 2025-12-31

Sucursales (6): ['Ambato-Centro', 'Cuenca-Este', 'Guayaquil-Centro', 'Manta-Puerto', 'Quito-Norte', 'Quito-Sur']
Categorías (5): ['Alimentos', 'Cuidado Personal', 'Electrónica', 'Hogar', 'Limpieza']

Transacciones con promoción (promo=1): 599,916
Transacciones sin promoción (promo=0): 1,400,084

────────────────────────────────────────────────────────────
Estadísticas descriptivas:


id_transaccion,ventas,precio_unitario,promo,descuento_pct
2000000.0,2000000.0,2000000.0,2000000.0,1800761.0
1000000.5,82.46998230499999,151.55103581999998,0.299958,6.755128526217527
577350.4135271751,44.742764253600974,85.69055491179397,0.45823935146127687,11.708662059351875
1.0,5.0,3.0,0.0,0.0
500000.75,43.7,77.37,0.0,0.0
1000000.5,82.44,151.61,0.0,0.0
1500000.25,121.2,225.69,1.0,10.9
2000000.0,160.0,300.0,1.0,40.0


In [0]:
print("🔍 ANÁLISIS DE VALORES NULOS")
print("=" * 60)
nulos = df.isnull().sum()
print(nulos[nulos > 0])
pct_nulos = (df['descuento_pct'].isnull().sum() / len(df)) * 100
print(f"\nPorcentaje de nulos en descuento_pct: {pct_nulos:.2f}%")
print("→ Los nulos en descuento_pct corresponden a transacciones sin promoción (promo=0)")

# Limpieza: rellenar nulos de descuento_pct con 0
df['descuento_pct'] = df['descuento_pct'].fillna(0)
print(f"\nNulos después de limpieza: {df.isnull().sum().sum()}")
print("✅ Datos limpios y listos para análisis")

🔍 ANÁLISIS DE VALORES NULOS
descuento_pct    199239
dtype: int64

Porcentaje de nulos en descuento_pct: 9.96%
→ Los nulos en descuento_pct corresponden a transacciones sin promoción (promo=0)

Nulos después de limpieza: 0
✅ Datos limpios y listos para análisis


In [0]:
# Ingeniería de características temporales
df['fecha'] = pd.to_datetime(df['fecha'])
df['mes'] = df['fecha'].dt.month
df['dia_semana'] = df['fecha'].dt.dayofweek  # 0=Lunes, 6=Domingo
df['semana'] = df['fecha'].dt.isocalendar().week.astype(int)
df['fin_de_semana'] = (df['dia_semana'] >= 5).astype(int)

dias_es = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves',
           4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
df['nombre_dia'] = df['dia_semana'].map(dias_es)

meses_es = {1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril', 5:'Mayo', 6:'Junio',
            7:'Julio', 8:'Agosto', 9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'}
df['nombre_mes'] = df['mes'].map(meses_es)

print("✅ Características creadas: mes, dia_semana, semana, fin_de_semana, nombre_dia, nombre_mes")
display(df.head(10))

✅ Características creadas: mes, dia_semana, semana, fin_de_semana, nombre_dia, nombre_mes


id_transaccion,fecha,sucursal,categoria,ventas,precio_unitario,promo,descuento_pct,mes,dia_semana,semana,fin_de_semana,nombre_dia,nombre_mes
1,2025-09-15T00:00:00.000Z,Quito-Norte,Hogar,61.93,141.22,0,0.0,9,0,38,0,Lunes,Septiembre
2,2025-10-10T00:00:00.000Z,Guayaquil-Centro,Cuidado Personal,122.08,31.42,1,39.6,10,4,41,0,Viernes,Octubre
3,2025-03-19T00:00:00.000Z,Guayaquil-Centro,Limpieza,143.58,25.14,1,28.4,3,2,12,0,Miércoles,Marzo
4,2025-11-30T00:00:00.000Z,Ambato-Centro,Alimentos,58.55,96.7,0,0.0,11,6,48,1,Domingo,Noviembre
5,2025-01-30T00:00:00.000Z,Guayaquil-Centro,Hogar,138.54,86.27,1,16.9,1,3,5,0,Jueves,Enero
6,2025-04-27T00:00:00.000Z,Ambato-Centro,Hogar,84.58,288.7,1,10.4,4,6,17,1,Domingo,Abril
7,2025-01-29T00:00:00.000Z,Cuenca-Este,Alimentos,139.81,264.09,1,11.6,1,2,5,0,Miércoles,Enero
8,2025-05-11T00:00:00.000Z,Ambato-Centro,Hogar,47.07,155.38,0,0.0,5,6,19,1,Domingo,Mayo
9,2025-07-25T00:00:00.000Z,Guayaquil-Centro,Alimentos,141.94,176.13,0,0.0,7,4,30,0,Viernes,Julio
10,2025-01-28T00:00:00.000Z,Ambato-Centro,Alimentos,14.63,238.08,1,11.5,1,1,5,0,Martes,Enero


In [0]:
# Muestra aleatoria para visualizaciones (mejor rendimiento con Plotly)
df_sample = df.sample(n=50000, random_state=42)

fig_cat = px.box(
    df_sample,
    x='categoria',
    y='ventas',
    color='categoria',
    title='📊 Distribución de Ventas por Categoría de Producto',
    labels={'categoria': 'Categoría', 'ventas': 'Ventas ($)'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig_cat.update_layout(showlegend=False)
fig_cat.show()

<!doctype html>

In [0]:
fig_suc = px.box(
    df_sample,
    x='sucursal',
    y='ventas',
    color='sucursal',
    title='📊 Distribución de Ventas por Sucursal',
    labels={'sucursal': 'Sucursal', 'ventas': 'Ventas ($)'},
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig_suc.update_layout(showlegend=False)
fig_suc.show()

<!doctype html>

In [0]:
# Agregar ventas semanales por sucursal
ventas_semanales = df.groupby(['semana', 'sucursal']).agg(
    ventas_total=('ventas', 'sum'),
    transacciones=('id_transaccion', 'count'),
    ventas_promedio=('ventas', 'mean')
).reset_index()

fig_ts = px.line(
    ventas_semanales,
    x='semana',
    y='ventas_total',
    color='sucursal',
    title='📈 Ventas Semanales por Sucursal (2025)',
    labels={'semana': 'Semana del Año', 'ventas_total': 'Ventas Totales ($)'},
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig_ts.update_layout(hovermode='x unified')
fig_ts.show()

<!doctype html>

In [0]:
df_sample['grupo_promo'] = np.where(df_sample['promo']==1, 'Con Promoción', 'Sin Promoción')

fig_promo = px.box(
    df_sample,
    x='grupo_promo',
    y='ventas',
    color='grupo_promo',
    title='📊 Distribución de Ventas: Con Promoción vs Sin Promoción',
    labels={'grupo_promo': 'Grupo', 'ventas': 'Ventas ($)'},
    color_discrete_map={'Con Promoción': '#2E86C1', 'Sin Promoción': '#E74C3C'}
)
fig_promo.update_layout(showlegend=False)
fig_promo.show()

<!doctype html>

## 🤖 Modelo de Regresión Lineal para Predicción de Ventas

Entrenamos un modelo de regresión lineal para predecir las ventas (`ventas`) utilizando las variables del dataset:

* **precio_unitario**: Precio unitario del producto
* **promo**: Indicador de promoción (0/1)
* **descuento_pct**: Porcentaje de descuento aplicado
* **sucursal**: Sucursal (codificación one-hot)
* **categoria**: Categoría de producto (codificación one-hot)
* **mes**: Mes de la transacción
* **dia_semana**: Día de la semana
* **fin_de_semana**: Indicador de fin de semana (0/1)

In [0]:
# Preparar características para el modelo de regresión
features_num = ['precio_unitario', 'promo', 'descuento_pct', 'mes', 'dia_semana', 'fin_de_semana']
features_cat = ['sucursal', 'categoria']

# One-Hot Encoding para variables categóricas
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoded_cat = encoder.fit_transform(df[features_cat])
encoded_cols = encoder.get_feature_names_out(features_cat)

df_encoded = pd.DataFrame(encoded_cat, columns=encoded_cols, index=df.index)
X = pd.concat([df[features_num], df_encoded], axis=1)
y = df['ventas']

print(f"Variables predictoras: {list(X.columns)}")
print(f"Total de features: {X.shape[1]}")
print(f"Total de observaciones: {X.shape[0]:,}")

# División train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nConjunto de entrenamiento: {X_train.shape[0]:,} registros")
print(f"Conjunto de prueba: {X_test.shape[0]:,} registros")

Variables predictoras: ['precio_unitario', 'promo', 'descuento_pct', 'mes', 'dia_semana', 'fin_de_semana', 'sucursal_Cuenca-Este', 'sucursal_Guayaquil-Centro', 'sucursal_Manta-Puerto', 'sucursal_Quito-Norte', 'sucursal_Quito-Sur', 'categoria_Cuidado Personal', 'categoria_Electrónica', 'categoria_Hogar', 'categoria_Limpieza']
Total de features: 15
Total de observaciones: 2,000,000

Conjunto de entrenamiento: 1,600,000 registros
Conjunto de prueba: 400,000 registros


In [0]:
# Entrenar modelo de regresión lineal
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Predicciones
y_pred_train = modelo.predict(X_train)
y_pred_test = modelo.predict(X_test)

# Métricas de evaluación
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_test = mean_absolute_error(y_test, y_pred_test)

print("=" * 60)
print("📈 RESULTADOS DEL MODELO DE REGRESIÓN LINEAL")
print("=" * 60)
print(f"\nR² (Entrenamiento): {r2_train:.4f}")
print(f"R² (Prueba):        {r2_test:.4f}")
print(f"RMSE (Prueba):      {rmse_test:.4f}")
print(f"MAE (Prueba):       {mae_test:.4f}")
print(f"\nIntercepto: {modelo.intercept_:.4f}")

# Gráfico de dispersión: valores reales vs predichos (muestra)
sample_idx = np.random.choice(len(y_test), size=5000, replace=False)
fig_scatter = px.scatter(
    x=y_test.iloc[sample_idx],
    y=y_pred_test[sample_idx],
    title='📊 Valores Reales vs Predichos (Muestra de 5,000)',
    labels={'x': 'Ventas Reales ($)', 'y': 'Ventas Predichas ($)'},
    opacity=0.5,
    color_discrete_sequence=['#2E86C1']
)
fig_scatter.add_trace(go.Scatter(
    x=[y_test.min(), y_test.max()],
    y=[y_test.min(), y_test.max()],
    mode='lines', name='Predicción Perfecta',
    line=dict(color='red', dash='dash')
))
fig_scatter.show()

📈 RESULTADOS DEL MODELO DE REGRESIÓN LINEAL

R² (Entrenamiento): 0.0000
R² (Prueba):        -0.0000
RMSE (Prueba):      44.7400
MAE (Prueba):       38.7554

Intercepto: 82.5910


<!doctype html>

In [0]:
# Análisis de coeficientes del modelo
coef_df = pd.DataFrame({
    'variable': X.columns,
    'coeficiente': modelo.coef_
}).sort_values('coeficiente', key=abs, ascending=False)

print("📊 Coeficientes del Modelo (ordenados por magnitud):")
print("=" * 60)
display(coef_df)

# Visualización de coeficientes
fig_coef = px.bar(
    coef_df.head(15),
    x='coeficiente',
    y='variable',
    orientation='h',
    title='🔝 Top 15 Coeficientes del Modelo de Regresión Lineal',
    labels={'coeficiente': 'Coeficiente', 'variable': 'Variable'},
    color='coeficiente',
    color_continuous_scale='RdBu_r'
)
fig_coef.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_coef.show()

📊 Coeficientes del Modelo (ordenados por magnitud):


variable,coeficiente
sucursal_Quito-Norte,0.16865270079651143
categoria_Cuidado Personal,0.16812900782330878
promo,0.14084842828885677
sucursal_Quito-Sur,-0.13120451008237893
fin_de_semana,0.12321481146048646
categoria_Limpieza,-0.10574658073102422
sucursal_Cuenca-Este,-0.08863238396626578
categoria_Hogar,-0.07666075371961278
sucursal_Guayaquil-Centro,-0.06592901067709783
dia_semana,-0.03461518522905617


<!doctype html>

## 🔬 Pruebas de Hipótesis: Impacto de las Promociones en las Ventas

### Prueba T de Student

**Hipótesis Nula (H₀)**: No hay diferencia significativa en las ventas entre transacciones con y sin promoción.
**Hipótesis Alternativa (H₁)**: Las promociones influyen significativamente en las ventas.
**Nivel de significancia**: α = 0.05

In [0]:
# Prueba T de Student: Ventas con promoción vs sin promoción
g_promo = df[df['promo']==1]['ventas']
g_regular = df[df['promo']==0]['ventas']

# Estadísticas descriptivas por grupo
mean_p, std_p, n_p = g_promo.mean(), g_promo.std(), g_promo.count()
mean_r, std_r, n_r = g_regular.mean(), g_regular.std(), g_regular.count()
diff_mean = mean_p - mean_r

# Prueba T (Welch's t-test, varianzas desiguales)
t_stat, p_val = stats.ttest_ind(g_promo, g_regular, equal_var=False)

# Intervalo de confianza del 95%
se_diff = np.sqrt((std_p**2/n_p) + (std_r**2/n_r))
ci_low, ci_high = stats.t.interval(0.95, df=min(n_p-1, n_r-1), loc=diff_mean, scale=se_diff)

print("=" * 60)
print("🔬 PRUEBA T DE STUDENT: PROMOCIÓN vs SIN PROMOCIÓN")
print("=" * 60)
print(f"\nGrupo Con Promoción:    N={n_p:,} | Media=${mean_p:.2f} | Std=${std_p:.2f}")
print(f"Grupo Sin Promoción:    N={n_r:,} | Media=${mean_r:.2f} | Std=${std_r:.2f}")
print(f"\nDiferencia observada de medias: ${diff_mean:.2f}")
print(f"Estadístico t: {t_stat:.4f}")
print(f"p-valor: {p_val:.4e}")
print(f"Intervalo de Confianza 95%: [${ci_low:.2f}, ${ci_high:.2f}]")

if p_val < 0.05:
    print(f"\n✅ Se rechaza la hipótesis nula (p < 0.05)")
    print("→ Existe evidencia estadísticamente significativa de que las promociones influyen en las ventas.")
else:
    print(f"\n❌ No se rechaza la hipótesis nula (p ≥ 0.05)")
    print("→ No existe evidencia estadística de que las promociones influyan en las ventas.")

🔬 PRUEBA T DE STUDENT: PROMOCIÓN vs SIN PROMOCIÓN

Grupo Con Promoción:    N=599,916 | Media=$82.50 | Std=$44.71
Grupo Sin Promoción:    N=1,400,084 | Media=$82.46 | Std=$44.76

Diferencia observada de medias: $0.04
Estadístico t: 0.6056
p-valor: 5.4475e-01
Intervalo de Confianza 95%: [$-0.09, $0.18]

❌ No se rechaza la hipótesis nula (p ≥ 0.05)
→ No existe evidencia estadística de que las promociones influyan en las ventas.


In [0]:
fig_t = px.box(
    df_sample,
    x='grupo_promo',
    y='ventas',
    color='grupo_promo',
    title='📊 Comparación de Ventas: Con Promoción vs Sin Promoción',
    labels={'grupo_promo': 'Grupo', 'ventas': 'Ventas ($)'},
    color_discrete_map={'Con Promoción': '#2E86C1', 'Sin Promoción': '#E74C3C'}
)
fig_t.update_layout(showlegend=False)
fig_t.show()

# Histograma comparativo
fig_hist = px.histogram(
    df_sample,
    x='ventas',
    color='grupo_promo',
    nbins=50,
    title='📊 Distribución de Ventas por Grupo',
    labels={'ventas': 'Ventas ($)', 'count': 'Frecuencia'},
    color_discrete_map={'Con Promoción': '#2E86C1', 'Sin Promoción': '#E74C3C'},
    opacity=0.7
)
fig_hist.update_layout(barmode='overlay')
fig_hist.show()

<!doctype html>

<!doctype html>

In [0]:
# Prueba Chi-Cuadrado: Asociación entre categoría de producto y promoción
tabla_contingencia = pd.crosstab(df['categoria'], df['promo'])
print("📊 Tabla de Contingencia: Categoría vs Promoción")
print("=" * 60)
display(tabla_contingencia)

chi2_stat, p_val_chi2, dof, expected = stats.chi2_contingency(tabla_contingencia)
print(f"\n{'='*60}")
print("🔬 PRUEBA CHI-CUADRADO: CATEGORÍA vs PROMOCIÓN")
print(f"{'='*60}")
print(f"\nEstadístico Chi²: {chi2_stat:.4f}")
print(f"p-valor: {p_val_chi2:.4e}")
print(f"Grados de Libertad: {dof}")

if p_val_chi2 < 0.05:
    print(f"\n✅ Se rechaza la hipótesis nula (p < 0.05)")
    print("→ Existe asociación significativa entre la categoría de producto y la presencia de promociones.")
else:
    print(f"\n❌ No se rechaza la hipótesis nula (p ≥ 0.05)")
    print("→ No existe asociación significativa entre la categoría de producto y las promociones.")

📊 Tabla de Contingencia: Categoría vs Promoción


0,1
279999,120477
280448,119678
280514,119599
279542,119986
279581,120176



🔬 PRUEBA CHI-CUADRADO: CATEGORÍA vs PROMOCIÓN

Estadístico Chi²: 6.0354
p-valor: 1.9652e-01
Grados de Libertad: 4

❌ No se rechaza la hipótesis nula (p ≥ 0.05)
→ No existe asociación significativa entre la categoría de producto y las promociones.


In [0]:
fig_chi2 = px.histogram(
    df_sample,
    x='categoria',
    color='grupo_promo',
    barmode='group',
    title='📊 Distribución de Promociones por Categoría de Producto',
    labels={'categoria': 'Categoría', 'count': 'Cantidad de Transacciones', 'grupo_promo': 'Grupo'},
    color_discrete_map={'Con Promoción': '#2E86C1', 'Sin Promoción': '#E74C3C'}
)
fig_chi2.show()

<!doctype html>

# 📌 Conclusiones y Recomendaciones

## Hallazgos Principales

### 1. Preparación de Datos
* Se procesaron **2,000,000 transacciones** de Comercial Andina correspondientes al año 2025.
* El dataset cuenta con **6 sucursales** (Quito-Norte, Quito-Sur, Guayaquil-Centro, Cuenca-Este, Ambato-Centro, Manta-Puerto) y **5 categorías** de producto (Alimentos, Electrónica, Hogar, Limpieza, Cuidado Personal).
* Se imputaron los valores nulos de `descuento_pct` con 0 (correspondientes a transacciones sin promoción).
* Se crearon características temporales: mes, día de la semana, semana ISO, fin de semana.

### 2. Modelo de Regresión Lineal
* El modelo de regresión lineal obtuvo un **R² ≈ 0.0000**, lo que indica que las variables predictoras disponibles (precio_unitario, promo, descuento_pct, sucursal, categoría, variables temporales) **no explican la variabilidad de las ventas** de forma lineal.
* **RMSE = 44.74** y **MAE = 38.76**, valores altos en relación al rango de ventas ($5–$160).
* Los coeficientes más relevantes fueron `sucursal_Quito-Norte` (+0.17), `categoria_Cuidado Personal` (+0.17) y `promo` (+0.14), todos de magnitud muy pequeña.

### 3. Prueba de Hipótesis: Promociones
* **Prueba T de Student**: t = 0.6056, p-valor = 0.5448 (> 0.05).
* La diferencia de medias entre transacciones con promoción ($82.50) y sin promoción ($82.46) es de apenas **$0.04**.
* **Conclusión**: No se rechaza la hipótesis nula. **Las promociones no influyen significativamente** en el monto de las ventas.
* **Prueba Chi-Cuadrado**: Chi² = 6.0354, p-valor = 0.1965 (> 0.05). No existe asociación significativa entre la categoría de producto y la presencia de promociones.

---

## Recomendaciones para Optimizar Decisiones Comerciales

1. **Rediseñar la estrategia de promociones**: Las promociones actuales no generan un impacto significativo en las ventas. Se recomienda:
   * Segmentar promociones por categoría y sucursal con descuentos diferenciados.
   * Probar descuentos de mayor magnitud o promociones tipo 2x1 en categorías específicas.
   * Medir el impacto con pruebas A/B antes de escalar.

2. **Incorporar variables adicionales al modelo**: El R² ≈ 0 sugiere que faltan variables explicativas. Se recomienda incluir:
   * Volumen de unidades vendidas por transacción.
   * Datos de inventario y stock disponible.
   * Información de clientes (segmento, frecuencia de compra).
   * Variables macroeconómicas (estacionalidad, festivos, inflación).

3. **Explorar modelos no lineales**: La regresión lineal no captura la complejidad del comportamiento de ventas. Probar:
   * Árboles de decisión y Random Forest.
   * Gradient Boosting (XGBoost, LightGBM).
   * Modelos de series temporales (Prophet, ARIMA) para el pronóstico semanal.

4. **Análisis por sucursal y categoría**: Las sucursales Quito-Norte y Quito-Sur muestran los mayores coeficientes positivos y negativos respectivamente. Se recomienda:
   * Investigar las prácticas comerciales de Quito-Norte como benchmark positivo.
   * Diagnosticar las causas del bajo desempeño relativo de Quito-Sur.

5. **Monitoreo de ventas semanales**: La visualización de ventas semanales por sucursal permite:
   * Optimizar la gestión de inventario por temporada.
   * Anticipar demanda en semanas de mayor volumen.
   * Ajustar personal y recursos logísticos según patrones estacionales.